In [3]:
# ==========================================================
# PROJECT PATH SETUP
# ==========================================================

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project Root:", PROJECT_ROOT)

Project Root: d:\Kavya Shankar_Workspace\Kaggle Projects\Predicting Smartphone Addiction


In [4]:
# ==========================================================
# PHASE 04 | PREPROCESSING
# STEP 00 | LOAD + FEATURE ENGINEERING
# ==========================================================

from src.data_loader import load_data
from src.feature_engineering import engineer_features

train_df, test_df, sample_submission = load_data()

train_fe = engineer_features(train_df)
test_fe = engineer_features(test_df)

print("Train Shape:", train_fe.shape)
print("Test Shape :", test_fe.shape)

Train Shape: (691369, 23)
Test Shape : (296302, 22)


In [5]:
# ==========================================================
# PHASE 04 | PREPROCESSING
# STEP 01 | SEPARATE FEATURES AND TARGET
# ==========================================================

X = train_fe.drop(columns=["addicted_label"])
y = train_fe["addicted_label"]

X_test = test_fe.copy()

print("X Shape      :", X.shape)
print("y Shape      :", y.shape)
print("X_test Shape :", X_test.shape)

print("\nTarget in X     :", "addicted_label" in X.columns)
print("Target in X_test:", "addicted_label" in X_test.columns)

X Shape      : (691369, 22)
y Shape      : (691369,)
X_test Shape : (296302, 22)

Target in X     : False
Target in X_test: False


In [6]:
from src.data_loader import load_data
from src.feature_engineering import engineer_features

print("Imports successful")

Imports successful


In [7]:
# ==========================================================
# STEP 02 | STRATIFIED TRAIN / VALIDATION SPLIT
# ==========================================================

from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train :", X_train.shape)
print("X_valid :", X_valid.shape)
print("y_train :", y_train.shape)
print("y_valid :", y_valid.shape)

X_train : (553095, 22)
X_valid : (138274, 22)
y_train : (553095,)
y_valid : (138274,)


In [8]:
print("Original:")
print(y.value_counts(normalize=True))

print("\nTrain:")
print(y_train.value_counts(normalize=True))

print("\nValidation:")
print(y_valid.value_counts(normalize=True))

Original:
addicted_label
1    0.709424
0    0.290576
Name: proportion, dtype: float64

Train:
addicted_label
1    0.709424
0    0.290576
Name: proportion, dtype: float64

Validation:
addicted_label
1    0.709425
0    0.290575
Name: proportion, dtype: float64


In [9]:
# ==========================================================
# STEP 03 | IDENTIFY FEATURE TYPES
# ==========================================================

numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

Numerical Features:
['id', 'age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time', 'total_entertainment_hours', 'weekend_difference', 'entertainment_ratio', 'social_media_ratio', 'gaming_ratio', 'work_study_ratio', 'leisure_to_work_ratio', 'screen_time_waking_ratio', 'non_screen_time']

Categorical Features:
['gender', 'stress_level', 'academic_work_impact']


In [10]:
print("\nNumerical Count   :", len(numerical_features))
print("Categorical Count :", len(categorical_features))


Numerical Count   : 19
Categorical Count : 3


In [11]:
# ==========================================================
# STEP 04 | REMOVE IDENTIFIER
# ==========================================================

X_train = X_train.drop(columns=["id"])
X_valid = X_valid.drop(columns=["id"])
X_test = X_test.drop(columns=["id"])

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("X_test :", X_test.shape)

X_train: (553095, 21)
X_valid: (138274, 21)
X_test : (296302, 21)


In [12]:
print("id in X_train:", "id" in X_train.columns)
print("id in X_valid:", "id" in X_valid.columns)
print("id in X_test :", "id" in X_test.columns)

id in X_train: False
id in X_valid: False
id in X_test : False


In [13]:
test_ids = test_fe["id"].copy()

print(test_ids.head())

0    691369
1    691370
2    691371
3    691372
4    691373
Name: id, dtype: int64


In [14]:
# ==========================================================
# STEP 05 | CHECK MISSING VALUES
# ==========================================================

missing_train = X_train.isna().sum()
missing_train = missing_train[missing_train > 0].sort_values(ascending=False)

missing_valid = X_valid.isna().sum()
missing_valid = missing_valid[missing_valid > 0].sort_values(ascending=False)

print("TRAIN MISSING VALUES")
print(missing_train)

print("\n" + "=" * 60)

print("VALIDATION MISSING VALUES")
print(missing_valid)

TRAIN MISSING VALUES
entertainment_ratio          192219
leisure_to_work_ratio        191726
total_entertainment_hours    165812
social_media_ratio           151417
gaming_ratio                 146866
weekend_difference           138809
work_study_ratio             110342
social_media_hours           107287
screen_time_waking_ratio     105668
non_screen_time              105668
gaming_hours                 101432
weekend_screen_time           89719
daily_screen_time_hours       76757
app_opens_per_day             64354
notifications_per_day         54060
stress_level                  44116
work_study_hours              41107
sleep_hours                   35556
academic_work_impact          35259
gender                        23236
age                           23103
dtype: int64

VALIDATION MISSING VALUES
entertainment_ratio          47904
leisure_to_work_ratio        47822
total_entertainment_hours    41276
social_media_ratio           37692
gaming_ratio                 36780
weekend_

In [15]:
print("\nTotal missing values")
print("Train:", X_train.isna().sum().sum())
print("Valid:", X_valid.isna().sum().sum())
print("Test :", X_test.isna().sum().sum())


Total missing values
Train: 2004513
Valid: 500801
Test : 1040547


In [16]:
# ==========================================================
# STEP 06 | DEFINE PREPROCESSING FEATURE GROUPS
# ==========================================================

numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical Features:")
for feature in numerical_features:
    print("-", feature)

print("\nCategorical Features:")
for feature in categorical_features:
    print("-", feature)

print("\nNumerical Count   :", len(numerical_features))
print("Categorical Count :", len(categorical_features))


Numerical Features:
- age
- daily_screen_time_hours
- social_media_hours
- gaming_hours
- work_study_hours
- sleep_hours
- notifications_per_day
- app_opens_per_day
- weekend_screen_time
- total_entertainment_hours
- weekend_difference
- entertainment_ratio
- social_media_ratio
- gaming_ratio
- work_study_ratio
- leisure_to_work_ratio
- screen_time_waking_ratio
- non_screen_time

Categorical Features:
- gender
- stress_level
- academic_work_impact

Numerical Count   : 18
Categorical Count : 3


In [17]:
# ==========================================================
# STEP 07 | BUILD PREPROCESSOR
# ==========================================================

from src.preprocessing import build_preprocessor

preprocessor = build_preprocessor(
    numerical_features=numerical_features,
    categorical_features=categorical_features,
)

print("Preprocessor created successfully.")

Preprocessor created successfully.


In [18]:
print(preprocessor)

ColumnTransformer(transformers=[('numerical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['age', 'daily_screen_time_hours',
                                  'social_media_hours', 'gaming_hours',
                                  'work_study_hours', 'sleep_hours',
                                  'notifications_per_day', 'app_opens_per_day',
                                  'weekend_screen_time',
                                  'total_entertainment_hours',
                                  'weekend_difference', 'entertainment_ratio',
                                  'social_media_ratio', 'gaming_ratio',
                                  'work_study_ratio', 'leisure_to_work_ratio',
                                  'screen_time_waking_ratio',
                                  'non_screen_time']),
                                ('categorical',
  

In [19]:
# ==========================================================
# STEP 08 | FIT PREPROCESSOR ON TRAINING DATA
# ==========================================================

X_train_processed = preprocessor.fit_transform(X_train)

print("X_train processed shape:", X_train_processed.shape)

X_train processed shape: (553095, 26)


In [20]:
# ==========================================================
# STEP 09 | TRANSFORM VALIDATION AND TEST DATA
# ==========================================================

X_valid_processed = preprocessor.transform(X_valid)
X_test_processed = preprocessor.transform(X_test)

print("X_train processed:", X_train_processed.shape)
print("X_valid processed:", X_valid_processed.shape)
print("X_test processed :", X_test_processed.shape)

X_train processed: (553095, 26)
X_valid processed: (138274, 26)
X_test processed : (296302, 26)


In [21]:
# ==========================================================
# STEP 10 | VALIDATE PROCESSED DATA
# ==========================================================

import numpy as np

print("NaN values:")
print("Train:", np.isnan(X_train_processed.data).sum())
print("Valid:", np.isnan(X_valid_processed.data).sum())
print("Test :", np.isnan(X_test_processed.data).sum())

print("\nInfinite values:")
print("Train:", np.isinf(X_train_processed.data).sum())
print("Valid:", np.isinf(X_valid_processed.data).sum())
print("Test :", np.isinf(X_test_processed.data).sum())

NaN values:
Train: 0
Valid: 0
Test : 0

Infinite values:
Train: 0
Valid: 0
Test : 0


In [22]:
# ==========================================================
# STEP 11 | CHECK PROCESSED FEATURE NAMES
# ==========================================================

processed_feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(processed_feature_names))

print("\nProcessed Features:")
for i, feature in enumerate(processed_feature_names, start=1):
    print(f"{i:02d}. {feature}")

Number of processed features: 26

Processed Features:
01. numerical__age
02. numerical__daily_screen_time_hours
03. numerical__social_media_hours
04. numerical__gaming_hours
05. numerical__work_study_hours
06. numerical__sleep_hours
07. numerical__notifications_per_day
08. numerical__app_opens_per_day
09. numerical__weekend_screen_time
10. numerical__total_entertainment_hours
11. numerical__weekend_difference
12. numerical__entertainment_ratio
13. numerical__social_media_ratio
14. numerical__gaming_ratio
15. numerical__work_study_ratio
16. numerical__leisure_to_work_ratio
17. numerical__screen_time_waking_ratio
18. numerical__non_screen_time
19. categorical__gender_Female
20. categorical__gender_Male
21. categorical__gender_Other
22. categorical__stress_level_High
23. categorical__stress_level_Low
24. categorical__stress_level_Medium
25. categorical__academic_work_impact_No
26. categorical__academic_work_impact_Yes


In [23]:
print("\nProcessed matrix shape :", X_train_processed.shape)
print("Feature name count     :", len(processed_feature_names))


Processed matrix shape : (553095, 26)
Feature name count     : 26
